# Training Feature Extraction Pipeline

This notebook extracts facial features from the training dataset for the AI Interview Analyzer.

**Features Extracted:**
- Eye Aspect Ratio (EAR) - left, right, and average
- Blink detection flag
- Smile ratio
- Eye synchronization

**Input:** Images from `data/raw/train/` organized by emotion categories  
**Output:** CSV file with extracted features saved to `data/processed/training_features.csv`

**Architecture:**
- MediaPipe Face Landmarker for facial landmark detection
- Custom feature engineering functions
- Batch processing across all training images

In [4]:
import os 
import cv2 as cv
import numpy as np  
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [5]:
import math

def euclidean(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def smile_ratio(face_landmarks, w, h):
    # Get landmark points
    left = face_landmarks[61]
    right = face_landmarks[291]
    upper = face_landmarks[13]
    lower = face_landmarks[14]

    # Convert to pixel coordinates
    left_pt = (int(left.x * w), int(left.y * h))
    right_pt = (int(right.x * w), int(right.y * h))
    upper_pt = (int(upper.x * w), int(upper.y * h))
    lower_pt = (int(lower.x * w), int(lower.y * h))

    # Calculate width and height
    width = euclidean(left_pt, right_pt)
    height = euclidean(upper_pt, lower_pt)

    if height == 0:
        return 0

    return width / height

In [6]:
# =========================================
# Head pose landmarker 
# =========================================
NOSE_TIP = 1
CHIN = 152
LEFT_EYE_OUTER = 33
RIGHT_EYE_OUTER = 263
LEFT_MOUTH = 61
RIGHT_MOUTH = 291

In [7]:


def estimate_head_pose(face_landmarks, w, h):

    image_points = np.array([
        [face_landmarks[NOSE_TIP].x * w, face_landmarks[NOSE_TIP].y * h],
        [face_landmarks[CHIN].x * w, face_landmarks[CHIN].y * h],
        [face_landmarks[LEFT_EYE_OUTER].x * w, face_landmarks[LEFT_EYE_OUTER].y * h],
        [face_landmarks[RIGHT_EYE_OUTER].x * w, face_landmarks[RIGHT_EYE_OUTER].y * h],
        [face_landmarks[LEFT_MOUTH].x * w, face_landmarks[LEFT_MOUTH].y * h],
        [face_landmarks[RIGHT_MOUTH].x * w, face_landmarks[RIGHT_MOUTH].y * h],
    ], dtype="double")

    model_points = np.array([
        (0.0, 0.0, 0.0),         # Nose
        (0.0, -330.0, -65.0),    # Chin
        (-225.0, 170.0, -135.0), # Left eye
        (225.0, 170.0, -135.0),  # Right eye
        (-150.0, -150.0, -125.0),# Left mouth
        (150.0, -150.0, -125.0)  # Right mouth
    ])

    focal_length = w
    center = (w / 2, h / 2)

    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ], dtype="double")

    dist_coeffs = np.zeros((4,1))

    success, rotation_vec, translation_vec = cv.solvePnP(
        model_points,
        image_points,
        camera_matrix,
        dist_coeffs,
        flags=cv.SOLVEPNP_ITERATIVE
    )

    rmat, _ = cv.Rodrigues(rotation_vec)

    angles, _, _, _, _, _ = cv.RQDecomp3x3(rmat)

    pitch, yaw, roll = angles

    return pitch, yaw, roll

In [8]:
# ========================================
# Facial Landmark Constants
# ========================================
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
LEFT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH = [61, 291, 13, 14]  # left, right, upper, lower


# ========================================
# Helper Functions
# ========================================
def _pt(face_landmarks, idx, w, h):
    """Convert normalized landmark to pixel coordinates."""
    lm = face_landmarks[idx]
    return np.array([lm.x * w, lm.y * h], dtype=np.float32)


def eye_aspect_ratio(face_landmarks, eye_idx, w, h):
    """
    Calculate Eye Aspect Ratio (EAR).
    
    EAR = (||p2-p6|| + ||p3-p5||) / (2 * ||p1-p4||)
    
    Args:
        face_landmarks: MediaPipe face landmarks
        eye_idx: List of 6 landmark indices for the eye
        w, h: Image width and height
    
    Returns:
        float: Eye aspect ratio
    """
    p1 = _pt(face_landmarks, eye_idx[0], w, h)
    p2 = _pt(face_landmarks, eye_idx[1], w, h)
    p3 = _pt(face_landmarks, eye_idx[2], w, h)
    p4 = _pt(face_landmarks, eye_idx[3], w, h)
    p5 = _pt(face_landmarks, eye_idx[4], w, h)
    p6 = _pt(face_landmarks, eye_idx[5], w, h)

    vertical_1 = np.linalg.norm(p2 - p6)
    vertical_2 = np.linalg.norm(p3 - p5)
    horizontal = np.linalg.norm(p1 - p4)

    if horizontal == 0:
        return 0.0
    return float((vertical_1 + vertical_2) / (2.0 * horizontal))


def smile_ratio(face_landmarks, w, h):
    """
    Calculate smile ratio (mouth width / mouth height).
    
    Args:
        face_landmarks: MediaPipe face landmarks
        w, h: Image width and height
    
    Returns:
        float: Smile ratio
    """
    left = _pt(face_landmarks, MOUTH[0], w, h)
    right = _pt(face_landmarks, MOUTH[1], w, h)
    upper = _pt(face_landmarks, MOUTH[2], w, h)
    lower = _pt(face_landmarks, MOUTH[3], w, h)

    mouth_width = np.linalg.norm(left - right)
    mouth_height = np.linalg.norm(upper - lower)

    if mouth_height == 0:
        return 0.0
    return float(mouth_width / mouth_height)

In [9]:
# ==========================
# eye gaze direction 
# ==========================

gaze_map = {
    "looking_left": -1,
    "looking_center": 0,
    "looking_right": 1
}

# gaze_value = gaze_map[gaze_direction]
def estimate_gaze_direction(face_landmarks, w, h):

    left_eye_center_x = np.mean([face_landmarks[i].x * w for i in LEFT_EYE])
    right_eye_center_x = np.mean([face_landmarks[i].x * w for i in RIGHT_EYE])

    face_center = (face_landmarks[1].x * w)

    gaze_offset = (left_eye_center_x + right_eye_center_x)/2 - face_center

    if gaze_offset > 10:
        return 1
    elif gaze_offset < -10:
        return -1
    else:
        return 0

In [13]:
# ========================================
# Configuration
# ========================================
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
TRAIN_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "train")
OUTPUT_CSV = r"ai-interview-analyzer\data\processed\training_features.csv"
MODEL_PATH = r"face_landmarker.task"

# EAR threshold for blink detection (not used in static images, but kept for consistency)
EAR_THRESHOLD = 0.25

# ========================================
# Path Resolution
# ========================================
# Resolve training data directory
train_dir = os.path.abspath(TRAIN_DIR)
if not os.path.isdir(train_dir):
    alt_dir = os.path.abspath(os.path.join("..", "data", "raw", "train"))
    if os.path.isdir(alt_dir):
        train_dir = alt_dir
    else:
        raise FileNotFoundError(f"Train folder not found: {train_dir}")

# Resolve model path
model_path = os.path.abspath(MODEL_PATH)
if not os.path.isfile(model_path):
    # Try alternative paths
    alt_paths = [
        os.path.join("..", MODEL_PATH),
        r"E:\mediapipe\face_landmarker.task",
    ]
    for alt in alt_paths:
        if os.path.isfile(alt):
            model_path = alt
            break
    else:
        raise FileNotFoundError(f"MediaPipe model not found: {model_path}")

# Create output directory if needed
output_csv = os.path.abspath(OUTPUT_CSV)
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

print(f"Training data directory: {train_dir}")
print(f"MediaPipe model: {model_path}")
print(f"Output CSV: {output_csv}")


# ========================================
# Initialize MediaPipe Face Landmarker (ONCE)
# ========================================
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,  # Assume one face per interview frame
)

landmarker = FaceLandmarker.create_from_options(options)
print("MediaPipe Face Landmarker initialized successfully.\n")


# ========================================
# Feature Extraction Pipeline
# ========================================
def extract_features_from_image(image_path, landmarker):
    """
    Extract facial features from a single image.
    
    Args:
        image_path: Path to the image file
        landmarker: MediaPipe FaceLandmarker instance
    
    Returns:
        dict or None: Extracted features, or None if no face detected
    """
    # Load image
    image = cv.imread(image_path)
    if image is None:
        print(f"Warning: Could not load image: {image_path}")
        return None
    image = cv.resize(image, (640, 480))
    h, w, _ = image.shape
    
    # Convert to RGB for MediaPipe
    rgb_image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
    
    # Detect face landmarks
    result = landmarker.detect(mp_image)
    
    if not result.face_landmarks or len(result.face_landmarks) == 0:
        print(f"Warning: No face detected in {os.path.basename(image_path)}")
        return None
    
    # Extract features from the first detected face
    face_landmarks = result.face_landmarks[0]
    
    # Compute Eye Aspect Ratios
    left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE, w, h)
    right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE, w, h)
    avg_ear = (left_ear + right_ear) / 2.0
    
    # Compute eye synchronization (difference between left and right EAR)
    eye_sync = abs(left_ear - right_ear)
    
    # Compute smile ratio
    smile = smile_ratio(face_landmarks, w, h)
    
    # Blink flag (0 for static images, set based on EAR if needed)
    blink_flag = 0
    pitch, yaw, roll = estimate_head_pose(face_landmarks, w, h)
    eye_gaze = estimate_gaze_direction(face_landmarks, w,h)
    
    return {
        'image_name': os.path.basename(image_path),
        'left_ear': left_ear,
        'right_ear': right_ear,
        'avg_ear': avg_ear,
        'blink_flag': blink_flag,
        'smile_ratio': smile,
        'eye_sync': eye_sync,
        'head_pitch': pitch,
        'head_yaw': yaw,
        'head_roll': roll,
        'eye_gaze_direction' :eye_gaze
    }


# ========================================
# Process All Training Images
# ========================================
features_list = []
processed_count = 0
failed_count = 0

# Get all subdirectories (emotion categories)
emotion_dirs = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]

print(f"Found {len(emotion_dirs)} emotion categories: {emotion_dirs}\n")
print("Extracting features from training images...")

for emotion in emotion_dirs:
    emotion_path = os.path.join(train_dir, emotion)
    image_files = [f for f in os.listdir(emotion_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    print(f"\nProcessing {emotion}: {len(image_files)} images")
    
    for image_file in image_files:
        image_path = os.path.join(emotion_path, image_file)
        
        # Extract features
        features = extract_features_from_image(image_path, landmarker)
        
        if features is not None:
            features['emotion'] = emotion  # Add emotion label
            features_list.append(features)
            processed_count += 1
        else:
            failed_count += 1
        
        # Progress update every 100 images
        if (processed_count + failed_count) % 100 == 0:
            print(f"  Processed: {processed_count + failed_count} images...")

print(f"\n{'='*50}")
print(f"Feature extraction complete!")
print(f"Successfully processed: {processed_count} images")
print(f"Failed/No face detected: {failed_count} images")
total_images = processed_count + failed_count

if total_images > 0:
    failure_rate = (failed_count / total_images) * 100
    print(f"Failure rate: {failure_rate:.2f}%")
print(f"{'='*50}\n")


# ========================================
# Save to CSV
# ========================================
if len(features_list) > 0:
    df = pd.DataFrame(features_list)
    
    # Reorder columns for better readability
    column_order = ['image_name', 'emotion', 'avg_ear', 'left_ear', 'right_ear', 
                    'blink_flag', 'smile_ratio', 'eye_sync','head_pitch','head_yaw','head_roll',"eye_gaze_direction"]
    df = df[column_order]
    
    df.to_csv(output_csv, index=False)
    print(f"✓ Features saved to: {output_csv}")
    print(f"✓ Dataset shape: {df.shape}")
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print("ERROR: No features extracted. Please check your training data.")
    
# Clean up MediaPipe resources safely
try:
    if 'landmarker' in locals() and landmarker is not None:
        landmarker.close()
        print("MediaPipe Face Landmarker closed.")
except Exception as e:
    print(f"Warning: Failed to close landmarker cleanly: {e}")

# Also save a copy to the requested absolute location
desired_output_dir = r"E:\resume_project\ai-interview-analyzer\data\processed"
os.makedirs(desired_output_dir, exist_ok=True)
desired_output_csv = os.path.join(desired_output_dir, os.path.basename(OUTPUT_CSV))

# Save only if a non-empty DataFrame exists
if 'df' in locals() and isinstance(df, pd.DataFrame) and not df.empty:
    # Avoid duplicate write if both paths resolve to the same file
    if os.path.abspath(desired_output_csv) != os.path.abspath(output_csv):
        df.to_csv(desired_output_csv, index=False)
        print(f"Additional CSV saved to: {desired_output_csv}")
    else:
        print("Additional CSV skipped (same as primary output path).")
else:
    print("Additional CSV skipped (no extracted feature DataFrame available).")

print("\nFeature extraction pipeline completed.")

Training data directory: e:\resume_project\ai-interview-analyzer\data\raw\train
MediaPipe model: E:\mediapipe\face_landmarker.task
Output CSV: e:\resume_project\ai-interview-analyzer\src\model_training\ai-interview-analyzer\data\processed\training_features.csv
MediaPipe Face Landmarker initialized successfully.

Found 7 emotion categories: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

Extracting features from training images...

Processing angry: 3995 images
  Processed: 100 images...
  Processed: 200 images...
  Processed: 300 images...
  Processed: 400 images...
  Processed: 500 images...
  Processed: 600 images...
  Processed: 700 images...
  Processed: 800 images...
  Processed: 900 images...
  Processed: 1000 images...
  Processed: 1100 images...
  Processed: 1200 images...
  Processed: 1300 images...
  Processed: 1400 images...
  Processed: 1500 images...
  Processed: 1600 images...
  Processed: 1700 images...
  Processed: 1800 images...
  Processed: 1900 i